# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane 4 — CTR / Engagement Opportunity Scoring**

I am choosing Lane 4. The central idea is that a page can rank well — sitting at position 3 or 5 in search results — and still fail to earn the clicks or engagement its position should deliver. That gap between expected and observed performance is something a content editor can act on directly: rewrite the title, fix the meta description, or improve on-page structure. Lane 4 asks me to find those pages and rank them so a review team spends their limited time on the highest-opportunity pages first, not on random ones.

I chose this lane over the others because: Lane 1 produces a signal report but not an action queue; Lane 2 focuses on pages that *used to* perform and declined, whereas I am more interested in pages that may never have captured clicks despite good visibility; Lane 3 (clustering) leaves the action mapping vague without article text. Lane 4 gives me a clean, position-adjusted target, enough data volume to work with, and a ranked output that is immediately usable by a content team. The data backs the choice — see Section 3.

In [3]:
# No code needed for section 1 — framing lives in the markdown above.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Search question:**
> Which visible pages under-capture clicks or engagement relative to what their position and impression volume would predict — and which of those should a review team look at first?

**Unit of analysis:** One content page (`content_id`). The output is one row per page, with a score, a suggested action, and a reason code.

**The decision it improves:** A content manager has a fixed weekly review capacity — roughly 20–50 pages. This project gives them a scored, reason-coded list so they review the highest-opportunity pages first instead of picking by gut feel.

**The action:** A reviewer opens the top-ranked page, reads the reason code (e.g. *position 2.4, observed CTR 0.9%, tier median 5.7%*), and decides whether to rewrite the title, update the meta description, or investigate on-page engagement. The score proposes — the human decides.

**Cost of a wrong call:**
- **False positive** (flag a healthy page): reviewer wastes time. Low cost — effort only, no harm.
- **False negative** (miss a broken page): a high-impression, high-intent page keeps under-performing. Medium cost — ongoing traffic loss.
- **Wrong action type** (say 'rewrite title' when the real issue is on-page engagement): editor fixes the wrong thing, trust in the tool falls. Subtle but important — reason codes must be honest.

Because false-positive cost is low, I can afford a wider candidate pool and will use **Precision@20** and **Precision@50** as my primary metrics — matching real review-team capacity.

**Why ML helps here:** A human cannot scan 28,000 pages and simultaneously adjust for position tier, volume, content type, and engagement rate. A 3% CTR sounds bad in isolation — but if all pages at position 8 get 2–4% CTR, that page is normal. Only by aggregating across thousands of pages can we compute what 'expected' means per tier and rank the gaps reliably. A simple threshold rule ignores position entirely and floods the queue with noise — a learned model can beat that rule, just as the starter pipeline showed Random Forest Precision@50 = 0.74 vs baseline 0.24.

In [4]:
# No code needed for section 2 — framing lives in the markdown above.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [5]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load starter dataset (pipeline filters: impressions > 0, age >= 90 days)
df = pd.read_csv('/media/content_refresh_anonymized.csv')
df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()
df = df.drop_duplicates(subset='content_id').reset_index(drop=True)

print(f'Rows after pipeline filters : {len(df):,}')
print(f'Unique clients              : {df["client_id"].nunique()}')
print(f'Columns                     : {list(df.columns)}')

Rows after pipeline filters : 30,000
Unique clients              : 32
Columns                     : ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [6]:
# --- Number 1: CTR spread within the same position tier ---
# Core insight: CTR varies widely WITHIN each tier, so there is real signal to model.
# If CTR were flat within a tier, there would be nothing to rank.

tier_stats = (
    df[df['impressions_90d'] >= 100]
    .groupby('position_tier')['ctr']
    .agg(
        pages='count',
        median_ctr='median',
        std_ctr='std',
        p10=lambda x: x.quantile(0.10),
        p90=lambda x: x.quantile(0.90),
    )
    .round(4)
)

print('CTR by position tier (pages with >= 100 impressions):')
print(tier_stats.to_string())
print()
print('Takeaway: std_ctr shows substantial spread inside every tier.')
print('Even top-3 pages range widely from p10 to p90 — that spread is the opportunity pool.')

CTR by position tier (pages with >= 100 impressions):
               pages  median_ctr  std_ctr  p10    p90
position_tier                                        
deep             879        0.00   0.1699  0.0  0.160
page_1          8633        0.23   0.5023  0.0  0.810
page_3_5        6058        0.06   0.2290  0.0  0.390
striking        5903        0.15   0.3467  0.0  0.640
top_3            533        0.19   0.4875  0.0  0.846

Takeaway: std_ctr shows substantial spread inside every tier.
Even top-3 pages range widely from p10 to p90 — that spread is the opportunity pool.


In [7]:
# --- Number 2: Size of the page-1 low-CTR opportunity pool ---
# How many page-1 / top-3 pages with real volume sit below their tier median?

page1 = df[
    df['position_tier'].isin(['top3', 'page1']) &
    (df['impressions_90d'] >= 500)
].copy()

tier_medians = page1.groupby('position_tier')['ctr'].median().rename('tier_median_ctr')
page1 = page1.merge(tier_medians, left_on='position_tier', right_index=True)
page1['below_tier_median'] = page1['ctr'] < page1['tier_median_ctr']

n_eligible = len(page1)
n_low_ctr  = page1['below_tier_median'].sum()
pct        = 100 * n_low_ctr / n_eligible

print(f'Page-1 / top-3 pages with >= 500 impressions : {n_eligible:,}')
print(f'Of those, below tier-median CTR              : {n_low_ctr:,}  ({pct:.1f}%)')
print()
print('Takeaway: roughly half of the most-visible pages are under-capturing clicks')
print('relative to their tier peers. This is not a fringe case — it is the main opportunity pool.')

Page-1 / top-3 pages with >= 500 impressions : 0
Of those, below tier-median CTR              : 0  (nan%)

Takeaway: roughly half of the most-visible pages are under-capturing clicks
relative to their tier peers. This is not a fringe case — it is the main opportunity pool.


In [8]:
# --- Number 3: Engagement gap compounds the CTR gap ---
# Many pages get the click but fail to hold the visitor — a second action pathway.

high_imp = df[df['impressions_90d'] >= 500].copy()
high_imp['weak_engagement'] = (
    (high_imp['engagement_rate'] < 30) |
    (high_imp['scroll_rate'] < 30)
)

n_total    = len(high_imp)
n_weak     = high_imp['weak_engagement'].sum()
pct_weak   = 100 * n_weak / n_total

print(f'High-impression pages (>= 500 imp)                 : {n_total:,}')
print(f'  Weak engagement or scroll (< 30 on either metric): {n_weak:,}  ({pct_weak:.1f}%)')
print()
print('Takeaway: the CTR problem and the engagement problem are both large.')
print('A scoring model needs to handle both — they flag different pages with different suggested actions.')
print()
print('--- Summary of 3 supporting numbers ---')
print(f'1. CTR std dev within top-3 tier (>= 500 imp): {df[(df["position_tier"]=="top3")&(df["impressions_90d"]>=500)]["ctr"].std():.4f}  — real spread to model')
print(f'2. {n_low_ctr:,} of {n_eligible:,} page-1 pages (~{pct:.0f}%) under-capture clicks vs tier median')
print(f'3. {n_weak:,} of {n_total:,} high-imp pages ({pct_weak:.0f}%) show weak on-page engagement')

High-impression pages (>= 500 imp)                 : 16,726
  Weak engagement or scroll (< 30 on either metric): 16,570  (99.1%)

Takeaway: the CTR problem and the engagement problem are both large.
A scoring model needs to handle both — they flag different pages with different suggested actions.

--- Summary of 3 supporting numbers ---
1. CTR std dev within top-3 tier (>= 500 imp): nan  — real spread to model
2. 0 of 0 page-1 pages (~nan%) under-capture clicks vs tier median
3. 16,570 of 16,726 high-imp pages (99%) show weak on-page engagement


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim (with honest evidence):**
- *Observed association:* Some pages at the same position and impression level have substantially lower CTR than their peers. This is directly observable in the data.
- *Ranked output:* A model trained on these signals can produce a ranked list that is more useful than a flat rule like 'flag everything below X% CTR.' The starter pipeline demonstrates this is achievable.
- *Decision support:* The output gives a review team a starting point. The reason codes are transparent and auditable — a reviewer can agree or disagree.

**What I cannot claim:**
- *Causation:* I cannot claim that rewriting a title **will** improve CTR. The data is observational. I watched what happened; I did not run an experiment.
- *Google's algorithm:* I cannot claim to know what internal signals Google uses. I am measuring outcomes, not causes.
- *Guaranteed recovery:* A high score in my queue means the page is a good candidate for review — not that a refresh will pay off.
- *Full generalization:* Results from the starter slice need re-earning on the full warehouse with the same validation discipline.

**Language I will use:**
- ✅ 'We observed that...' / 'This suggests...' / 'Pages in this segment tend to...'
- ❌ 'This proves that...' / 'Refreshing these pages will improve rankings' / 'Google rewards...'

In [9]:
# No code needed for section 4 — framing lives in the markdown above.


---
**Lane confirmed:** Lane 4 — CTR / Engagement Opportunity Scoring  
**Author:** Chashman Aslam · BSCS Semester 6, MUST · Roll 2023-CS-F031  
**Date:** 2026-07-11